In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
from datetime import date
import plotly.graph_objects as go
from datetime import datetime, time

from utils import compute_gex_df, fmt

### Ticker

In [14]:
TICKER = "QQQ"

ticker = yf.Ticker(TICKER)
spot = ticker.fast_info["last_price"]
expirations = ticker.options

In [15]:
print(f"Spot price: ${spot:.2f}\n")
print(f"Available expirations ({len(expirations)}):\n")
for e in expirations[:10]:
    print(f"  {e}")

Spot price: $729.45

Available expirations (30):

  2026-05-27
  2026-05-28
  2026-05-29
  2026-06-01
  2026-06-02
  2026-06-03
  2026-06-04
  2026-06-05
  2026-06-12
  2026-06-18


> pick first (today) expiration 0DTE

In [16]:
exp = expirations[0]

chain = ticker.option_chain(exp)

calls = chain.calls
puts  = chain.puts

print(f"Expiration: {exp}")
print(f"Calls: {len(calls)} rows   Puts: {len(puts)} rows")
print(f"\nColumns: {list(calls.columns)}")

Expiration: 2026-05-27
Calls: 102 rows   Puts: 103 rows

Columns: ['contractSymbol', 'lastTradeDate', 'strike', 'lastPrice', 'bid', 'ask', 'change', 'percentChange', 'volume', 'openInterest', 'impliedVolatility', 'inTheMoney', 'contractSize', 'currency']


In [17]:
print("=== CALLS (first 5) ===")
calls[["strike","openInterest","impliedVolatility","volume","bid","ask"]].head()

# print("\n=== PUTS (first 5) ===")
# print(puts[["strike","openInterest","impliedVolatility","volume","bid","ask"]].head())

=== CALLS (first 5) ===


,strike,openInterest,impliedVolatility,volume,bid,ask
0,525.0,29,3.606446,29,202.5,206.37
1,540.0,59,2.101567,59,187.5,191.53
2,570.0,0,0.500005,4,157.5,161.40
3,575.0,2,1.695314,2,152.5,156.53
4,595.0,15,2.425785,15,132.5,136.37


### Data cleanliness

In [18]:
print("=== CALLS quality check ===")
print(f"  OI = 0 or NaN:  {(calls['openInterest'].fillna(0) == 0).sum()} / {len(calls)} rows")
print(f"  IV = 0 or NaN:  {(calls['impliedVolatility'].fillna(0) == 0).sum()} / {len(calls)} rows")

print("\n=== PUTS quality check ===")
print(f"  OI = 0 or NaN:  {(puts['openInterest'].fillna(0) == 0).sum()} / {len(puts)} rows")
print(f"  IV = 0 or NaN:  {(puts['impliedVolatility'].fillna(0) == 0).sum()} / {len(puts)} rows")

=== CALLS quality check ===
  OI = 0 or NaN:  1 / 102 rows
  IV = 0 or NaN:  0 / 102 rows

=== PUTS quality check ===
  OI = 0 or NaN:  2 / 103 rows
  IV = 0 or NaN:  0 / 103 rows


### Compute gamma => GEX for each strike

In [19]:
exp_date = date.fromisoformat(exp)

market_close = datetime.combine(exp_date, time(16, 0))  # 4pm ET
now = datetime.now()  # assumes you're running in ET
minutes_left = max((market_close - now).total_seconds() / 60, 5)
T = minutes_left / (252 * 390)  # 390 trading minutes per day

In [20]:
RATE = 0.0368  # risk-free rate

calls_gex = compute_gex_df(calls, spot, T, +1, RATE)
puts_gex  = compute_gex_df(puts,  spot, T, -1, RATE)

print(f"Calls GEX rows: {len(calls_gex)}")
print(f"Puts GEX rows: {len(puts_gex)}")

print("\nCalls sample:")
calls_gex.head(3)

Calls GEX rows: 101
Puts GEX rows: 101

Calls sample:


,strike,oi,iv,gamma,gex
0,525.0,29.0,3.606446,5.729943e-38,1.661683e-34
1,540.0,59.0,2.101567,1.257602e-89,7.419851e-86
2,575.0,2.0,1.695314,3.448709e-86,6.897418e-84


### Net Gex

In [21]:
combined = pd.concat([calls_gex, puts_gex])

gex_by_strike = combined.groupby("strike")["gex"].sum().reset_index()
gex_by_strike = gex_by_strike.sort_values("strike").reset_index(drop=True)

print(f"Total strikes: {len(gex_by_strike)}")
print(f"Net GEX total: {gex_by_strike['gex'].sum():,.0f}")
gex_by_strike.head()

Total strikes: 119
Net GEX total: 118,288


,strike,gex
0,495.0,-5.878068e-141
1,500.0,-6.619582e-134
2,505.0,-3.203689e-135
3,515.0,-5.365180e-136
4,520.0,-1.428867e-138


### Plot

In [22]:
# ── Config ───────────────────────────────────────────────────────────────────
N = 50  # number of strikes to show (centered around spot)

# ── Filter to N nearest strikes ──────────────────────────────────────────────
gex_plot = gex_by_strike.copy()
gex_plot["dist"] = (gex_plot["strike"] - spot).abs()
gex_plot = (gex_plot.nsmallest(N, "dist")
                    .sort_values("strike")
                    .reset_index(drop=True))

strikes  = gex_plot["strike"].values
gex_vals = gex_plot["gex"].values

# ── Key levels ────────────────────────────────────────────────────────────────
sign_changes = np.where(np.diff(np.sign(gex_vals)))[0]
if len(sign_changes):
    idx = sign_changes[np.argmin(np.abs(strikes[sign_changes] - spot))]
    x0, x1 = strikes[idx], strikes[idx + 1]
    y0, y1 = gex_vals[idx], gex_vals[idx + 1]
    gamma_flip = x0 - y0 * (x1 - x0) / (y1 - y0)
else:
    gamma_flip = strikes[np.argmin(np.abs(gex_vals))]

call_wall_idx = np.argmax(gex_vals)
put_wall_idx  = np.argmin(gex_vals)
call_wall = strikes[call_wall_idx]
put_wall  = strikes[put_wall_idx]

print(f"Gamma Flip : ${gamma_flip:.1f}")
print(f"Call Wall  : ${call_wall:.0f}  (GEX {fmt(gex_vals[call_wall_idx])})")
print(f"Put Wall   : ${put_wall:.0f}  (GEX {fmt(gex_vals[put_wall_idx])})")

# ── Colours ───────────────────────────────────────────────────────────────────
bar_colors = ["#00e5a0" if v >= 0 else "#ff4560" for v in gex_vals]

COLORS = {
    "spot":       "#ffffff",
    "gamma_flip": "#facc15",
    "call_wall":  "#00e5a0",
    "put_wall":   "#ff4560",
}

# ── Traces (bars + one scatter per level for legend toggling) ─────────────────
fig = go.Figure()

# GEX bars (index 0)
fig.add_trace(go.Bar(
    x=strikes, y=gex_vals,
    marker_color=bar_colors, marker_line_width=0,
    name="Net GEX",
    showlegend=False,
    hovertemplate="Strike: $%{x}<br>GEX: %{y:,.1f}<extra></extra>",
))

# Each level as a thin vertical scatter — toggling its visibility also
# controls the matching shape via a JS trick using updatemenus isn't needed;
# instead we use a full-height filled area with zero width (a thin bar trick),
# or simply a scatter with two points at ymin/ymax.
ymin = gex_vals.min() * 1.15
ymax = gex_vals.max() * 1.15

def level_trace(x_val, color, label, dash):
    return go.Scatter(
        x=[x_val, x_val], y=[ymin, ymax],
        mode="lines",
        line=dict(color=color, width=1.5, dash=dash),
        name=label,
        legendgroup=label,
        hovertemplate=f"{label}: ${x_val:.1f}<extra></extra>",
    )

fig.add_trace(level_trace(spot,       COLORS["spot"],       f"Spot ${spot:.0f}",            "dot"))
fig.add_trace(level_trace(gamma_flip, COLORS["gamma_flip"], f"Gamma Flip ${gamma_flip:.0f}", "dash"))
fig.add_trace(level_trace(call_wall,  COLORS["call_wall"],  f"Call Wall ${call_wall:.0f}",   "dashdot"))
fig.add_trace(level_trace(put_wall,   COLORS["put_wall"],   f"Put Wall ${put_wall:.0f}",     "dashdot"))

# ── Layout ────────────────────────────────────────────────────────────────────
fig.update_layout(
    template="plotly_dark",
    title=dict(
        text=f"<b>{TICKER} GEX by Strike</b>  |  0DTE {exp}  |  Spot ${spot:.2f}",
        font_size=15, x=0.01,
    ),
    xaxis=dict(
        title="Strike",
        tickprefix="$",
        tickangle=60,
        # dtick=5
        tickmode="array",
        tickvals=strikes,
    ),
    yaxis=dict(title="Net GEX (gamma × OI × 100)", range=[ymin, ymax]),
    bargap=0.15,
    plot_bgcolor="#0d1117",
    paper_bgcolor="#0d1117",
    font_color="#c9d1d9",
    legend=dict(
        orientation="h",
        yanchor="bottom", y=1.02,
        xanchor="left",   x=0,
        bgcolor="rgba(0,0,0,0.4)",
        bordercolor="rgba(255,255,255,0.1)",
        borderwidth=1,
    ),
    height=800,
)

fig.show()

Gamma Flip : $730.0
Call Wall  : $729  (GEX +112.6K)
Put Wall   : $731  (GEX -21.9K)
